In [1]:
import pandas as pd
import duckdb

In [2]:
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE TABLE paysim AS
SELECT *
FROM read_csv_auto('../data/paysim_transactions.csv');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
row_count = con.execute("""
SELECT COUNT(*) AS total_rows
FROM paysim;
""").df()

row_count

,total_rows
0,6362620


In [18]:
con.execute(
    """select type,
              count(*) as transaction_count,
              min(amount) as min_amount,
              max(amount) as max_amount,
              avg(amount) as avg_amount,
              sum(amount) as fraud_transaction_amount,
              sum(case when isFraud = 1 then amount else 0 end) as fraud_transaction_amount
       from paysim
       where type in ('TRANSFER', 'CASH_OUT')
       group by type
       order by fraud_transaction_amount desc"""
).df()

,type,transaction_count,min_amount,max_amount,avg_amount,fraud_transaction_amount,fraud_transaction_amount_1
0,TRANSFER,532909,2.6,92445516.64,910647.009645,4.852920e+11,6.067213e+09
1,CASH_OUT,2237500,0.0,10000000.00,176273.964346,3.944130e+11,5.989202e+09


In [19]:
con.execute("""
SELECT
    type,
    isFraud,
    COUNT(*) AS transaction_count,
    ROUND(AVG(amount), 2) AS avg_amount,
    ROUND(MIN(amount), 2) AS min_amount,
    ROUND(MAX(amount), 2) AS max_amount
FROM paysim
WHERE type IN ('TRANSFER', 'CASH_OUT')
GROUP BY type, isFraud
ORDER BY type, isFraud DESC;
""").df()

,type,isFraud,transaction_count,avg_amount,min_amount,max_amount
0,CASH_OUT,1,4116,1455102.59,0.00,10000000.00
1,CASH_OUT,0,2233384,173917.16,0.01,2847566.62
2,TRANSFER,1,4097,1480891.67,63.80,10000000.00
3,TRANSFER,0,528812,906229.01,2.60,92445516.64


In [20]:
con.execute("""
SELECT
    type,
    amount,
    isFraud,
    CASE
        WHEN type IN ('TRANSFER', 'CASH_OUT')
             AND amount >= 200000
        THEN 1
        ELSE 0
    END AS rule_flag
FROM paysim
LIMIT 20;
""").df()

,type,amount,isFraud,rule_flag
0,PAYMENT,9839.64,0,0
1,PAYMENT,1864.28,0,0
2,TRANSFER,181.00,1,0
3,CASH_OUT,181.00,1,0
4,PAYMENT,11668.14,0,0
5,PAYMENT,7817.71,0,0
6,PAYMENT,7107.77,0,0
7,PAYMENT,7861.64,0,0
8,PAYMENT,4024.36,0,0
9,DEBIT,5337.77,0,0


In [21]:
con.execute("""
WITH rule_results AS (
    SELECT
        type,
        amount,
        isFraud,
        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
                 AND amount >= 200000
            THEN 1
            ELSE 0
        END AS rule_flag
    FROM paysim
)

SELECT
    COUNT(*) AS total_transactions,
    SUM(rule_flag) AS flagged_transactions,
    SUM(isFraud) AS total_fraud_transactions,
    SUM(CASE WHEN rule_flag = 1 AND isFraud = 1 THEN 1 ELSE 0 END) AS true_positives,
    SUM(CASE WHEN rule_flag = 1 AND isFraud = 0 THEN 1 ELSE 0 END) AS false_positives,
    SUM(CASE WHEN rule_flag = 0 AND isFraud = 1 THEN 1 ELSE 0 END) AS false_negatives
FROM rule_results;
""").df()

,total_transactions,flagged_transactions,total_fraud_transactions,true_positives,false_positives,false_negatives
0,6362620,1197669.0,8213.0,5471.0,1192198.0,2742.0


In [22]:
con.execute("""
WITH rule_results AS (
    SELECT
        type,
        amount,
        isFraud,
        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
                 AND amount >= 200000
            THEN 1
            ELSE 0
        END AS rule_flag
    FROM paysim
),

rule_summary AS (
    SELECT
        COUNT(*) AS total_transactions,
        SUM(rule_flag) AS flagged_transactions,
        SUM(isFraud) AS total_fraud_transactions,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 1 THEN 1 ELSE 0 END) AS true_positives,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 0 THEN 1 ELSE 0 END) AS false_positives,
        SUM(CASE WHEN rule_flag = 0 AND isFraud = 1 THEN 1 ELSE 0 END) AS false_negatives
    FROM rule_results
)

SELECT
    total_transactions,
    flagged_transactions,
    total_fraud_transactions,
    true_positives,
    false_positives,
    false_negatives,
    ROUND(true_positives * 1.0 / total_fraud_transactions, 4) AS recall,
    ROUND(true_positives * 1.0 / flagged_transactions, 4) AS precision
FROM rule_summary;
""").df()

,total_transactions,flagged_transactions,total_fraud_transactions,true_positives,false_positives,false_negatives,recall,precision
0,6362620,1197669.0,8213.0,5471.0,1192198.0,2742.0,0.6661,0.0046


In [23]:
rule_performance = con.execute("""
WITH rule_results AS (
    SELECT
        type,
        amount,
        isFraud,
        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
                 AND amount >= 200000
            THEN 1
            ELSE 0
        END AS rule_flag
    FROM paysim
),

rule_summary AS (
    SELECT
        COUNT(*) AS total_transactions,
        SUM(rule_flag) AS flagged_transactions,
        SUM(isFraud) AS total_fraud_transactions,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 1 THEN 1 ELSE 0 END) AS true_positives,
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 0 THEN 1 ELSE 0 END) AS false_positives,
        SUM(CASE WHEN rule_flag = 0 AND isFraud = 1 THEN 1 ELSE 0 END) AS false_negatives
    FROM rule_results
)

SELECT
    total_transactions,
    flagged_transactions,
    total_fraud_transactions,
    true_positives,
    false_positives,
    false_negatives,
    ROUND(true_positives * 1.0 / total_fraud_transactions, 4) AS recall,
    ROUND(true_positives * 1.0 / flagged_transactions, 4) AS precision
FROM rule_summary;
""").df()

rule_performance.to_csv("../outputs/first_fraud_rule_results.csv", index=False)

rule_performance

,total_transactions,flagged_transactions,total_fraud_transactions,true_positives,false_positives,false_negatives,recall,precision
0,6362620,1197669.0,8213.0,5471.0,1192198.0,2742.0,0.6661,0.0046
